# Agent 5 - Scholarship Agent

**What Agent 5 does:** uses Agent 4's annual cost estimates and the student's profile
to find relevant scholarships, check eligibility and calculate the remaining funding
gap. It preserves a distinction between eligible recommendations, records requiring
review, and confirmed awards.

**Input:** `state["finance"]`; `state["profile"]` including nationality, target field,
degree, GPA and scale; annual budget and currency; Agent 7's merit score; optional
confirmed awards. The catalog is loaded directly from
`data/raw/agent5/AGENT5_FINAL_DATASET.csv` (9,673 uploaded records).
`AGENT5_TIER_A_VERIFIED.csv` (1,238 records) supplies the uploaded preferred subset.
Its "verified"/"live" labels are provenance metadata, not proof of current deadlines.

**Processing:** normalize explicit catalog fields, retain unresolved requirements,
exclude explicit eligibility failures and expired awards, rank by textual relevance,
and calculate `max(0, annual_cost - annual_budget - confirmed_compatible_awards)`.
Unconfirmed scholarship coverage never reduces the funding gap. Only explicitly stated
award periods and GPA scales are used; missing information is not guessed.

**Output:** a copied state with `state["scholarships"]`, one result per finance entry:
`university`, `program_name`, `currency`, `funding_gap`, `funding_needed`,
`confirmed_funding`, `recommendations`, `review_candidates`, and `eligibility_review`.
Recommendations include scholarship name, coverage, eligibility status and deadline.

**Accuracy target: at least 96%. Status: NOT MEASURED.** The uploaded files contain
scholarship records, not independently labelled student-scholarship decisions or
student funding outcomes. Catalog `tier`, `data_confidence_score`, `funding_type`
and admissions labels from Agent 2 are not substituted for those outcomes. The
notebook includes a separate grouped evaluation workflow for genuine outcome data.
Correctness checks and retrieval demonstrations are not predictive-accuracy claims.

### How the seven agents connect

| Agent | What it does | Input | Output / next consumer |
|---|---|---|---|
| 1 - Profile | Structures the student's academic profile and candidate tier. | Student form, academic/test scores; documents when extraction is connected. | `state["profile"]`, including GPA/scale, target degree/field and tier; used by Agents 2, 3, 5, 6 and 7. |
| 2 - University | Retrieves and reranks university/program candidates; estimates admission likelihood. | Profile, preferences and Agent 7 profile strength. | `matched_universities`, grouped by university with program scores, admission probabilities and tiers; used by Agent 3. |
| 3 - Program | Ranks specific programs against interests within Agent 2's shortlist. | `matched_universities`, profile interests/target field and program catalog. | `programs`: university, program, specialization, duration/mode, fit score, match score, probability and tier; used by Agents 4 and 6. |
| 4 - Financial | Estimates annual tuition and living costs and reports estimate quality. | `programs` plus target degree. | `finance`: university, program, annual tuition/living/total cost, currency, quality and explanation; used by Agent 5. |
| 5 - Scholarship | Retrieves scholarship records, checks explicit eligibility rules and calculates funding gaps. | `finance`, profile, annual budget, Agent 7 merit, confirmed awards and the uploaded scholarship catalog. | `scholarships`: per-program funding gap, confirmed funding, eligible recommendations and ranked records needing review; used by the final report. |
| 6 - Career | Advisory research/industry alignment with an optional independently trained classifier. | `programs`, profile career goal, program catalog; optional verified outcomes/reports. | `careers`: path, alignment score, illustrative roles and sourced placement statistics (unknown when absent). Implemented in `Agent6_Career_agent.ipynb`. |
| 7 - Extracurricular | Scores resume achievements, leadership/research evidence and SOP/LOR quality. | Resume/achievement evidence and SOP/LOR inputs. | `extracurricular`, including `profile_strength_score` on a 0-100 scale; used by Agents 2 and 5. |

The framework describes the intended system; this table does not claim unimplemented
or unconnected services are running. Existing notebook functions remain unchanged.


## 1. Local inputs and contracts

Run independently with Python, pandas, NumPy, and scikit-learn. No API keys, network downloads, or generated scholarship records are used. Provide DataFrames directly, or place optional CSVs under `data/raw/agent5/`; discovery searches the current directory and its ancestors.

**`scholarship_catalog.csv`:** one row per unique `scholarship_id`. Required columns: `scholarship_id`, `scholarship_name`, `currency`, `coverage_basis`, `amount_period`, `eligible_nationalities`, `eligible_fields`, `eligible_degrees`, `min_gpa`, `gpa_scale`, `deadline_policy`, `deadline`. Also provide either `coverage_percent` (0-100) or `amount` (nonnegative), not both. `coverage_basis` is `tuition` or `total`; `amount_period` must be `annual`. Eligibility lists use `|` or Python lists; explicit `all` means unrestricted, whereas a blank value means unknown. `min_gpa=0` explicitly means no GPA requirement. `deadline_policy` is `fixed` (ISO date) or `open` (no fixed deadline, verified from source). Optional: `university` (`all` or exact name), `description`, `source_url`, `min_profile_strength_score` (0-100; absent means no merit requirement). Verify current source rules before importing; absence of a university column means no university restriction.

**Student:** `state["profile"]` contains `nationality`, `field_of_study` (then `target_program`, then `major`), `target_degree`, `gpa`, `gpa_scale`, `budget_annual`, and `budget_currency`. GPA thresholds compare fractions of explicitly supplied maximum scales; proportional comparison is not an official international credential conversion. Use a verified common scale where institutional conversion rules differ. Merit uses Agent 7's **0-100** `profile_strength_score` from profile or `state["extracurricular"]`.

**Agent 4 costs:** `state["finance"]` contains `university`, `program_name`, `total_cost`, `tuition_annual`, `currency`, optional `cost_period` (default `annual`, matching Agent 4). Negative, missing, inconsistent, or nonannual costs are not used for gap calculations. Missing budget or incompatible currency makes the gap unknown.

**Confirmed awards:** optional `state["confirmed_awards"]` records require `scholarship_id`, `status="confirmed"`, exact `university` and `program_name`, `currency`, `amount_period="annual"`, `annual_amount`, and explicit Boolean `stackable`. Optional `exclusive_group` prevents combining awards from the same group. Only these confirmed amounts reduce funding gaps. The largest valid nonstackable award is compared with the compatible stackable bundle; conflicting duplicate IDs are withheld. Candidate scholarships show individual potential coverage and never count as guaranteed funding.

Exact degree restrictions are preserved: an `MS`-only award does not accept `MBA` or `MEng`. Explicit `masters` accepts these degree types. The optional Agent 7 score remains on its original 0-100 scale.

Legacy `state["preferences"]["budget_max_usd"]` is used only when `budget_period="annual"` is explicitly supplied; otherwise normalize it into `profile.budget_annual` and `profile.budget_currency` first. A total-program budget is never silently treated as annual. Cost provenance from Agent 4 is retained as `cost_data_quality`; the gap is a calculation from supplied costs and can inherit estimation uncertainty.


**Upstream budget contract:** Agent 2 now writes `budget_period="annual"` and
`budget_currency="USD"` when processing the existing `preferences.budget_max_usd`
field without a declared period. This is the explicit project convention for
legacy profiles; an explicitly monthly or total-program budget is preserved and
requires conversion before an annual gap is calculated. This convention does not
infer scholarship award periods or GPA scales.

In [1]:
from pathlib import Path
import math
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

CATALOG_PATH = None  # Optional explicit Path; otherwise search data/raw/agent5/.
LABELS_PATH = None
LABELS_ARE_REAL = False  # True only after verifying independently annotated outcomes.
TARGET_ACCURACY = 0.96

def find_agent5_file(filename, explicit=None):
    if explicit is not None:
        path = Path(explicit).expanduser()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    for root in (Path.cwd(), *Path.cwd().parents):
        for path in (root / 'data' / 'raw' / 'agent5' / filename, root / filename):
            if path.is_file():
                return path
    return None

catalog_path = find_agent5_file('scholarship_catalog.csv', CATALOG_PATH)
if catalog_path is None:
    catalog_path = find_agent5_file('AGENT5_FINAL_DATASET.csv')
if catalog_path is None:
    catalog_path = find_agent5_file('AGENT5_TIER_A_VERIFIED.csv')
preferred_path = find_agent5_file('AGENT5_TIER_A_VERIFIED.csv')
preferred_scholarship_ids = (set(pd.read_csv(preferred_path)['scholarship_id'].astype(str))
                             if preferred_path else set())
labels_path = find_agent5_file('agent5_labelled_outcomes.csv', LABELS_PATH)
uploaded_scholarship_records = pd.read_csv(catalog_path) if catalog_path else None
scholarship_catalog = None  # Normalized after the adapter below is defined.
agent5_labelled_outcomes = pd.read_csv(labels_path) if labels_path else None
print('Scholarship catalogue:', str(catalog_path) if catalog_path else 'not supplied')
print('Real labelled outcomes:', str(labels_path) if labels_path else 'not supplied')
print('Agent 5 predictive accuracy: NOT MEASURED until real labelled data is evaluated.')


Scholarship catalogue: D:\Users\asmita\MAJOR\Major_Project-2027\data\raw\agent5\AGENT5_FINAL_DATASET.csv
Real labelled outcomes: D:\Users\asmita\MAJOR\Major_Project-2027\data\raw\agent5\agent5_labelled_outcomes.csv
Agent 5 predictive accuracy: NOT MEASURED until real labelled data is evaluated.


## 2. Conservative eligibility and funding rules

Missing information produces `unknown`; an explicit failed requirement produces `ineligible`. Eligibility rules are never filled with permissive defaults. Amounts are compared only within the same declared currency and annual period. Percentage coverage uses the stated tuition/total basis and is capped at relevant known costs. `as_of` is injectable for reproducible deadline checks; otherwise the current date in Asia/Kolkata is used.


Demonyms such as Indian/India and American/United States are normalized. Field
aliases are normalized, but overlapping nonidentical fields (for example Computer
Science versus Computer Science and Engineering) produce `unknown` for review,
not an automatic rejection or automatic approval. Explicit degree mismatches,
expired deadlines and known country exclusions still reject a candidate.

In [2]:
# Deterministic aliases only; fuzzy resemblance never proves eligibility.
def normalize_field(value):
    text = re.sub(r"[^a-z0-9]+", " ", clean(value).casefold()).strip()
    aliases = {'cs': 'computer science', 'computer sciences': 'computer science',
               'cse': 'computer science and engineering',
               'computer science engineering': 'computer science and engineering',
               'business administration': 'business administration'}
    return aliases.get(text, text)


KNOWN_COUNTRIES = {
    'india', 'united states', 'united kingdom', 'canada', 'australia', 'china',
    'germany', 'france', 'ireland', 'netherlands', 'new zealand', 'japan',
    'south korea', 'pakistan', 'bangladesh', 'nepal', 'sri lanka', 'south africa',
    'turkey', 'brazil', 'mexico', 'italy', 'spain', 'switzerland', 'egypt',
    'jordan', 'greece', 'hungary', 'bulgaria'
}


def clean(value):
    if value is None or (not isinstance(value, (list, tuple, set)) and pd.isna(value)):
        return ''
    return str(value).strip()

def number(value):
    try:
        result = float(value)
        return result if math.isfinite(result) else None
    except (ValueError, TypeError):
        return None

def tokens(value):
    items = value if isinstance(value, (list, tuple, set)) else clean(value).split('|')
    return {clean(item).casefold() for item in items if clean(item)}

def degree(value):
    value = clean(value).casefold().replace('.', '')
    return {'msc': 'ms', 'master of science': 'ms', 'master': 'masters', "master's": 'masters',
            'master of business administration': 'mba', 'master of engineering': 'meng',
            'doctorate': 'phd', 'doctoral': 'phd', 'bsc': 'bs',
            "bachelor's": 'bachelors'}.get(value, value)

def intended_field(profile):
    return next((profile[key] for key in ('field_of_study', 'target_program', 'major')
                 if clean(profile.get(key))), None)

CATALOG_COLUMNS = {'scholarship_id', 'scholarship_name', 'currency', 'coverage_basis',
    'amount_period', 'eligible_nationalities', 'eligible_fields', 'eligible_degrees',
    'min_gpa', 'gpa_scale', 'deadline_policy', 'deadline'}

def validate_catalog(catalog):
    missing = CATALOG_COLUMNS - set(catalog.columns)
    if missing:
        raise ValueError('Catalogue missing columns: ' + ', '.join(sorted(missing)))
    if catalog['scholarship_id'].map(clean).eq('').any():
        raise ValueError('Every scholarship needs a nonblank scholarship_id.')
    if catalog['scholarship_id'].map(clean).duplicated().any():
        raise ValueError('Duplicate scholarship_id: reconcile catalogue records first.')
    if catalog['scholarship_name'].map(clean).eq('').any():
        raise ValueError('Every scholarship needs a nonblank scholarship_name.')
    return catalog.copy()

def eligible(scholarship, profile, finance, as_of):
    failed, unknown = [], []
    def requirement(label, allowed, actual, transform=lambda x: clean(x).casefold()):
        choices = tokens(allowed)
        if 'all' in choices:
            return
        if label == 'degree':
            normalized = {degree(item) for item in choices}
            if ('masters' in normalized and degree(actual) in {'ms', 'mba', 'meng'}
                    or 'bachelors' in normalized and degree(actual) in {'bs', 'ba', 'beng'}):
                return
        if not choices or not clean(actual):
            unknown.append(label)
        elif transform(actual) not in {transform(v) for v in choices}:
            # Different field names can overlap; absence of an exact spelling is
            # not evidence that an interdisciplinary degree is disallowed.
            if label == 'field':
                unknown.append('field equivalence requires review')
            elif label == 'nationality' and (
                    transform(actual) not in KNOWN_COUNTRIES
                    or not all(transform(value) in KNOWN_COUNTRIES for value in choices)):
                unknown.append('nationality equivalence requires review')
            else:
                failed.append(label)
    requirement('nationality', scholarship.get('eligible_nationalities'), profile.get('nationality'),
                lambda value: country_name(value) if 'country_name' in globals() else clean(value).casefold())
    requirement('field', scholarship.get('eligible_fields'), intended_field(profile), normalize_field)
    requirement('degree', scholarship.get('eligible_degrees'), profile.get('target_degree'), degree)
    if 'university' in scholarship:
        requirement('university', scholarship.get('university'), finance.get('university'))
    minimum = number(scholarship.get('min_gpa'))
    if minimum is None or minimum < 0:
        unknown.append('GPA requirement')
    elif minimum > 0:
        actual, actual_scale, minimum_scale = map(number, (
            profile.get('gpa'), profile.get('gpa_scale'), scholarship.get('gpa_scale')))
        if (actual is None or actual_scale is None or minimum_scale is None
                or actual_scale <= 0 or minimum_scale <= 0
                or not 0 <= actual <= actual_scale or minimum > minimum_scale):
            unknown.append('GPA or GPA scale')
        elif actual / actual_scale + 1e-12 < minimum / minimum_scale:
            failed.append('GPA cutoff')
    if 'min_profile_strength_score' in scholarship:
        minimum_merit = number(scholarship.get('min_profile_strength_score'))
        merit = number(profile.get('profile_strength_score'))
        if minimum_merit is None or not 0 <= minimum_merit <= 100:
            unknown.append('merit requirement')
        elif minimum_merit > 0:
            if merit is None or not 0 <= merit <= 100:
                unknown.append('merit score')
            elif merit < minimum_merit:
                failed.append('merit cutoff')
    policy = clean(scholarship.get('deadline_policy')).casefold()
    deadline_text = clean(scholarship.get('deadline'))
    deadline = (pd.to_datetime(deadline_text, errors='coerce', utc=True)
                if re.fullmatch(r'\d{4}-\d{2}-\d{2}', deadline_text) else pd.NaT)
    if policy == 'fixed':
        if pd.isna(deadline):
            unknown.append('deadline')
        elif deadline.date() < as_of:
            failed.append('expired deadline')
    elif policy != 'open':
        unknown.append('deadline policy')
    # These columns are added only by the uploaded-data adapter.
    if '_raw_rules_need_review' in scholarship and scholarship['_raw_rules_need_review']:
        unknown.append('source academic/English/other requirements need student-specific review')
    if clean(scholarship.get('_application_status')).casefold().startswith('closed'):
        failed.append('application marked closed/historical')
    required_work = number(scholarship.get('_work_experience_years'))
    if required_work is not None and required_work > 0:
        actual_work = number(profile.get('work_experience_years'))
        if actual_work is None or actual_work < 0:
            unknown.append('work experience')
        elif actual_work < required_work:
            failed.append('work experience cutoff')
    host = clean(scholarship.get('_host_country'))
    if host:
        actual_host = country_name(finance.get('country')) if 'country_name' in globals() else clean(finance.get('country')).casefold()
        if not actual_host:
            unknown.append('destination country')
        elif actual_host != host:
            failed.append('destination country')
    status = 'ineligible' if failed else ('unknown' if unknown else 'eligible')
    return status, failed + ['unknown: ' + item for item in unknown]

def compatible_cost(finance):
    total, tuition = number(finance.get('total_cost')), number(finance.get('tuition_annual'))
    if total is None or total < 0 or not clean(finance.get('currency')):
        return False
    return (clean(finance.get('cost_period', 'annual')).casefold() == 'annual'
            and (tuition is None or 0 <= tuition <= total))

def potential_coverage(scholarship, finance):
    if not compatible_cost(finance):
        return None, 'unknown or inconsistent annual cost'
    if clean(scholarship.get('currency')).upper() != clean(finance.get('currency')).upper():
        return None, 'currency mismatch'
    if clean(scholarship.get('amount_period')).casefold() != 'annual':
        return None, 'nonannual or missing award period'
    basis = clean(scholarship.get('coverage_basis')).casefold()
    if basis not in {'tuition', 'total'}:
        return None, 'unknown coverage basis'
    base = number(finance.get('tuition_annual' if basis == 'tuition' else 'total_cost'))
    if base is None or base < 0:
        return None, 'missing coverage basis amount'
    percent, amount = number(scholarship.get('coverage_percent')), number(scholarship.get('amount'))
    if (percent is None) == (amount is None):
        return None, 'provide exactly one of coverage_percent or amount'
    if percent is not None:
        if not 0 <= percent <= 100:
            return None, 'coverage percent outside 0-100'
        amount = base * percent / 100
    if amount < 0:
        return None, 'negative award amount'
    return min(float(amount), base, float(finance['total_cost'])), None

def confirmed_funding(awards, finance):
    groups, withheld, blocked_ids = {}, [], set()
    for award in awards:
        if clean(award.get('status')).casefold() != 'confirmed':
            continue
        if (not clean(award.get('university')) or not clean(award.get('program_name'))
                or award.get('university') != finance.get('university')
                or award.get('program_name') != finance.get('program_name')):
            continue
        key = clean(award.get('scholarship_id'))
        amount = number(award.get('annual_amount'))
        stackable = award.get('stackable')
        valid = (bool(key) and amount is not None and amount >= 0
            and isinstance(stackable, (bool, np.bool_))
            and clean(award.get('amount_period')).casefold() == 'annual'
            and bool(clean(finance.get('currency')))
            and clean(award.get('currency')).upper() == clean(finance.get('currency')).upper())
        if not valid:
            blocked_ids.add(key)
            withheld.append(key or 'unnamed award')
            continue
        signature = (amount, bool(stackable), clean(award.get('exclusive_group')))
        groups.setdefault(key, set()).add(signature)
    stack_groups, standalone = {}, []
    for key, values in groups.items():
        if key in blocked_ids:
            continue
        if len(values) != 1:
            withheld.append(key + ': conflicting duplicate amounts/rules')
            continue
        amount, stackable, exclusive = next(iter(values))
        if stackable:
            bundle_key = ('exclusive', exclusive) if exclusive else ('award', key)
            stack_groups[bundle_key] = max(stack_groups.get(bundle_key, 0), amount)
        else:
            standalone.append(amount)
    total = max([sum(stack_groups.values()), *standalone], default=0)
    return total, withheld

## 3. Retrieval and the Agent 4 -> Agent 5 handoff

`scholarship_agent_v2` accepts a real catalogue DataFrame and optionally an already loaded sentence-transformer-compatible `embed_model`; it never downloads a model. Eligibility filtering precedes ranking. Offline fallback is TF-IDF retrieval over descriptions, names, and eligible fields. Optional semantic embeddings improve relevance when a model is supplied. Neither retrieval score is a calibrated probability.

This local interface keeps the notebook runnable without Qdrant. In deployment, pass actual catalogue records retrieved from the scholarship collection through the same eligibility and funding checks. Records are never invented when a catalogue is absent. The returned copy writes `state["scholarships"]` without modifying the input state. Rejected/unknown rows remain visible under `eligibility_review`. `coverage_%` is percentage of total annual cost; `potential_annual_coverage` preserves the monetary interpretation even for tuition-only scholarships.


In [3]:
def relevance_scores(rows, profile, finance, embed_model=None):
    texts = [' '.join(clean(row.get(key)) for key in
             ('scholarship_name', 'description', 'eligible_fields')) for row in rows]
    query = ' '.join(clean(value) for value in (profile.get('interests_text'), profile.get('interests'),
        intended_field(profile), finance.get('program_name')))
    if not texts or not query.strip():
        return np.zeros(len(texts)), 'no relevance query'
    if embed_model is not None:
        vectors = np.asarray(embed_model.encode([query, *texts]), dtype=float)
        if vectors.ndim != 2 or len(vectors) != len(texts) + 1 or not np.isfinite(vectors).all():
            raise ValueError('embed_model returned invalid embeddings')
        return np.clip(cosine_similarity(vectors[:1], vectors[1:])[0], 0, 1), 'semantic'
    try:
        vectors = TfidfVectorizer(ngram_range=(1, 2), strip_accents='unicode').fit_transform([query, *texts])
        return cosine_similarity(vectors[:1], vectors[1:])[0], 'tfidf'
    except ValueError:
        return np.zeros(len(texts)), 'no usable vocabulary'

def scholarship_agent_v2(state, catalog=None, *, as_of=None, embed_model=None, top_k=10):
    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError('top_k must be a positive integer')
    as_of = (pd.Timestamp.now(tz='Asia/Kolkata').date() if as_of is None else pd.Timestamp(as_of).date())
    if pd.isna(as_of):
        raise ValueError('as_of must be a valid assessment date')
    records = [] if catalog is None else validate_catalog(catalog).to_dict('records')
    profile, results = dict(state.get('profile', {})), []
    if 'profile_strength_score' not in profile and isinstance(state.get('extracurricular'), dict):
        profile['profile_strength_score'] = state['extracurricular'].get('profile_strength_score')
    preferences = state.get('preferences', {})
    if ('budget_annual' not in profile
            and clean(preferences.get('budget_period')).casefold() == 'annual'
            and 'budget_max_usd' in preferences):
        profile['budget_annual'] = preferences['budget_max_usd']
        profile['budget_currency'] = 'USD'
    host_lookup = {clean(uni.get('university', uni.get('university_name'))): uni.get('country')
                   for uni in state.get('matched_universities', [])}
    for finance_item in state.get('finance', []):
        finance = dict(finance_item)
        if not clean(finance.get('country')):
            finance['country'] = host_lookup.get(clean(finance.get('university')))
        total, budget = number(finance.get('total_cost')), number(profile.get('budget_annual'))
        valid_budget = (budget is not None and budget >= 0
            and bool(clean(profile.get('budget_currency')))
            and clean(profile.get('budget_currency')).upper() == clean(finance.get('currency')).upper())
        confirmed, withheld = confirmed_funding(state.get('confirmed_awards', []), finance)
        gap_known = compatible_cost(finance) and valid_budget
        gap = max(0.0, total - budget - confirmed) if gap_known else None
        candidates, review, pending = [], [], []
        for row in records:
            status, reasons = eligible(row, profile, finance, as_of)
            coverage, coverage_issue = potential_coverage(row, finance)
            if status != 'eligible' or coverage_issue:
                review.append({'scholarship_id': clean(row['scholarship_id']),
                    'eligibility_match': status, 'reasons': reasons + ([coverage_issue] if coverage_issue else [])})
                if status != 'ineligible':
                    pending.append({**row, '_status': status,
                                    '_reasons': reasons + ([coverage_issue] if coverage_issue else [])})
                continue
            candidates.append({**row, '_coverage': coverage})
        scores, retrieval = relevance_scores(candidates, profile, finance, embed_model)
        recommendations = []
        for row, score in zip(candidates, scores):
            coverage = row['_coverage']
            recommendations.append({'scholarship_id': clean(row['scholarship_id']),
                'scholarship_name': clean(row['scholarship_name']), 'coverage_%':
                    (round(100 * coverage / total, 2) if total and total > 0 else 0.0),
                'coverage_basis': clean(row['coverage_basis']), 'eligibility_match': 'eligible',
                'deadline': clean(row['deadline']), 'potential_annual_coverage': round(coverage, 2),
                'currency': clean(finance.get('currency')).upper(), 'award_status': 'not_confirmed',
                'match_score': round(float(score), 6), 'source_url': clean(row.get('source_url'))})
        recommendations.sort(key=lambda row: (-row['match_score'], -row['potential_annual_coverage'], row['scholarship_id']))
        pending_scores, _ = relevance_scores(pending, profile, finance, embed_model)
        review_candidates = []
        for row, score in zip(pending, pending_scores):
            review_candidates.append({
                'scholarship_id': clean(row['scholarship_id']),
                'scholarship_name': clean(row['scholarship_name']),
                'eligibility_match': row['_status'], 'review_required': True,
                'reasons': row['_reasons'], 'match_score': round(float(score), 6),
                'source_url': clean(row.get('source_url')),
                'reported_amount': number(row.get('_reported_amount', row.get('amount'))),
                'reported_currency': clean(row.get('currency')),
                'amount_period': clean(row.get('amount_period')) or 'unknown',
                'reported_deadline': clean(row.get('_reported_deadline', row.get('deadline'))),
                'uploaded_tier_a': bool(row.get('_uploaded_tier_a', False)),
                'raw_rules': row.get('_raw_rules', {}),
                'award_status': 'not_confirmed'})
        review_candidates.sort(key=lambda row: (-row['match_score'], -int(row['uploaded_tier_a']), row['scholarship_id']))
        results.append({'university': finance.get('university'), 'program_name': finance.get('program_name'),
            'currency': clean(finance.get('currency')).upper(), 'funding_gap': round(gap, 2) if gap is not None else None,
            'funding_needed': gap > 0 if gap is not None else None,
            'cost_data_quality': finance.get('cost_data_quality', 'UNSPECIFIED'),
            'funding_status': 'calculated_from_supplied_costs' if gap_known else 'unknown: verify annual costs, budget and currency',
            'confirmed_funding': round(min(confirmed, total), 2) if compatible_cost(finance) else None,
            'withheld_awards': withheld, 'recommendations': recommendations[:top_k],
            'eligibility_review': review, 'review_candidates': review_candidates[:top_k], 'retrieval': retrieval,
            'catalog_counts': {'examined': len(records), 'fully_recommendable': len(recommendations),
                               'needs_review': len(review_candidates),
                               'ineligible': sum(item['eligibility_match'] == 'ineligible' for item in review)},
            'catalogue_status': 'supplied' if catalog is not None else 'not supplied', 'as_of': as_of.isoformat()})
    return {**state, 'scholarships': results, 'status': 'scholarship_done'}


scholarship_agent = scholarship_agent_v2

## 3.1 Using your uploaded scholarship files

The adapter reads your files without editing them. It maps `eligible_countries`,
`degree_level`, `field_of_study`, `gpa_requirement`, `deadline` and source links into
the notebook's schema. Only unambiguous full dates are parsed. "Not specified" and
complex country/university conditions remain unknown; they are not unrestricted.
Both the raw text and original amount are preserved for review.

Per your instruction, a GPA scale is used only if an explicit `gpa_scale` column
or source text such as "on a 4.0 scale" supplies it. An award period is used only
if explicitly provided in `amount_period`, or the exact award amount is followed
by "per year", "/year", or "annually" in the source text. A generic mention of a
monthly stipend does not prove the period of the separate `amount` column.
Derived `coverage_percentage_clean` is not interpreted as verified annual coverage:
the uploaded file does not consistently state its denominator or period.

`review_candidates` makes incomplete but relevant records useful without certifying
eligibility. Rows marked historical/closed or with a past fixed deadline are excluded
from both recommendation lists. Source descriptions still require a current check.
The uploaded Tier A subset is not a held-out test set: its IDs overlap the full file.

In [4]:
def country_name(value):
    text = clean(value).casefold()
    return {'usa': 'united states', 'us': 'united states', 'u.s.': 'united states',
            'uk': 'united kingdom', 'u.k.': 'united kingdom',
            'indian': 'india', 'american': 'united states', 'british': 'united kingdom',
            'canadian': 'canada', 'australian': 'australia', 'chinese': 'china',
            'german': 'germany', 'french': 'france', 'irish': 'ireland',
            'pakistani': 'pakistan', 'bangladeshi': 'bangladesh', 'nepali': 'nepal',
            'dutch': 'netherlands', 'japanese': 'japan'}.get(text, text)


def explicit_date(value):
    text = clean(value)
    if not re.fullmatch(r'(?:\d{4}-\d{2}-\d{2}|\d{1,2}\s+[A-Za-z]+\s+\d{4})', text):
        return ''
    parsed = pd.to_datetime(text, errors='coerce', dayfirst=not text[:4].isdigit())
    return '' if pd.isna(parsed) else parsed.date().isoformat()


def explicit_gpa_scale(row, text):
    provided = number(row.get('gpa_scale'))
    if provided is not None and provided > 0:
        return provided
    patterns = [r'(?:on\s+(?:a\s+)?)?(\d+(?:\.\d+)?)\s*[- ]point\s+(?:GPA\s+)?scale',
                r'on\s+(?:a\s+)?(\d+(?:\.\d+)?)\s+scale',
                r'(?:GPA|grade point average).{0,45}?out of\s+(\d+(?:\.\d+)?)']
    found = {float(match) for pattern in patterns for match in re.findall(pattern, text, flags=re.I)}
    return next(iter(found)) if len(found) == 1 and next(iter(found)) > 0 else np.nan


def explicit_award_period(row, text):
    provided = clean(row.get('amount_period')).casefold()
    if provided:
        return provided
    amount = number(row.get('amount'))
    if amount is None or amount <= 0:
        return ''
    # Match the actual amount, not an unrelated annual cost or monthly stipend.
    forms = {f'{amount:g}', f'{amount:,.0f}' if amount.is_integer() else f'{amount:,.2f}'}
    exact = '(?:' + '|'.join(re.escape(form) for form in forms) + ')'
    pattern = r'(?<![\d.,])' + exact + r'(?:\.00)?\s*(?:/\s*year|per\s+year|annually|a\s+year)\b'
    return 'annual' if re.search(pattern, text, re.I) else ''


def adapt_uploaded_catalog(raw, preferred_ids=None):
    if CATALOG_COLUMNS.issubset(raw.columns):
        return validate_catalog(raw)
    required = {'scholarship_id', 'scholarship_name', 'eligible_countries',
                'degree_level', 'field_of_study'}
    if required - set(raw.columns):
        raise ValueError('Unrecognized uploaded catalog schema: ' + ', '.join(sorted(required - set(raw.columns))))
    preferred_ids = preferred_scholarship_ids if preferred_ids is None else set(preferred_ids)
    result = []
    for record in raw.to_dict('records'):
        row = dict(record)
        text = ' '.join(clean(record.get(key)) for key in
                        ('eligibility_text', 'description', 'academic_requirements', 'tuition_coverage', 'other_requirements'))
        nationality = clean(record.get('eligible_countries'))
        if nationality.casefold() in {'global', 'all', 'all countries'}:
            nationalities = 'all'
        elif re.search(r'[()]|commonwealth|developing|selected|priority|territories', nationality, re.I):
            nationalities = ''
        else:
            nationalities = '|'.join(country_name(item) for item in nationality.split(';') if clean(item))
        fields = clean(record.get('field_of_study'))
        if fields.casefold() in {'all', 'all fields', 'any field', 'all disciplines'}:
            fields = 'all'
        elif fields.casefold() in {'', 'not specified', 'unspecified'} or '(' in fields:
            fields = ''
        else:
            fields = '|'.join(item.strip() for item in fields.split(';'))
        degree_map = {'master': 'masters', 'masters': 'masters', 'bachelor': 'bachelors',
                      'bachelors': 'bachelors', 'undergraduate': 'bachelors', 'phd': 'phd',
                      'ms': 'ms', 'msc': 'ms', 'meng': 'meng', 'mba': 'mba', 'bs': 'bs'}
        raw_degrees = [item.strip().casefold() for item in clean(record.get('degree_level')).split(';')]
        degrees = '|'.join(degree_map[item] for item in raw_degrees if item in degree_map)
        university = clean(record.get('university'))
        if university.casefold() in {'all', 'any university'}:
            university = 'all'
        elif university.casefold() in {'global/international', 'not specified', 'unspecified'} or re.search(r'any |participating|partner', university, re.I):
            university = ''
        elif '_' in university:
            university = ''  # Institutional slugs need an authoritative name mapping.
        deadline = explicit_date(record.get('deadline'))
        host = clean(record.get('host_country'))
        if re.search(r'unspecified|various|multiple|overseas|[()]', host, re.I):
            host = ''
        raw_rules = {key: clean(record.get(key)) for key in
                     ('eligible_countries', 'degree_level', 'field_of_study', 'university',
                      'gpa_requirement', 'english_requirement', 'academic_requirements',
                      'other_requirements', 'work_experience_years', 'eligibility_text')}
        unresolved = any(raw_rules[key] for key in ('english_requirement', 'academic_requirements', 'other_requirements', 'eligibility_text'))
        row.update(
            eligible_nationalities=nationalities, eligible_fields=fields, eligible_degrees=degrees,
            university=university, min_gpa=number(record.get('gpa_requirement')),
            gpa_scale=explicit_gpa_scale(record, text),
            deadline=deadline, deadline_policy='fixed' if deadline else '',
            amount_period=explicit_award_period(record, text),
            coverage_basis=clean(record.get('coverage_basis')),
            coverage_percent=np.nan,  # No assumption about the denominator of derived percentages.
            source_url=clean(record.get('source_url')) or clean(record.get('application_url')),
            _reported_amount=record.get('amount'), _reported_deadline=record.get('deadline'),
            _uploaded_tier_a=str(record['scholarship_id']) in preferred_ids,
            _raw_rules=raw_rules, _raw_rules_need_review=unresolved,
            _work_experience_years=record.get('work_experience_years'),
            _application_status=record.get('application_status'),
            _host_country=country_name(host),
        )
        result.append(row)
    return validate_catalog(pd.DataFrame(result))


if uploaded_scholarship_records is not None:
    scholarship_catalog = adapt_uploaded_catalog(uploaded_scholarship_records)
    print('Uploaded rows loaded:', len(scholarship_catalog))
    print('Unique scholarship IDs:', scholarship_catalog['scholarship_id'].nunique())
    print('Uploaded preferred-subset records:', sum(scholarship_catalog.get('_uploaded_tier_a', pd.Series(False, index=scholarship_catalog.index))))
    print('Explicit GPA scales:', scholarship_catalog['gpa_scale'].notna().sum())
    print('Explicit annual award periods:', scholarship_catalog['amount_period'].eq('annual').sum())
    print('Unambiguous fixed deadlines:', scholarship_catalog['deadline_policy'].eq('fixed').sum())
    print('Catalog loaded for retrieval; student-level predictive accuracy is still NOT MEASURED.')
else:
    print('No scholarship catalog supplied.')

Uploaded rows loaded: 9673
Unique scholarship IDs: 9673
Uploaded preferred-subset records: 1238
Explicit GPA scales: 18
Explicit annual award periods: 0
Unambiguous fixed deadlines: 131
Catalog loaded for retrieval; student-level predictive accuracy is still NOT MEASURED.


## 4. Optional real-label comparison with untouched test students

Supply `agent5_labelled_outcomes.csv`, or call `evaluate_agent5_real_labels(frame, labels_are_real=True)`. Required columns: unique `record_id`, `student_id`, `label_source` (nonblank provenance), binary `needs_financial_support`, and numeric `total_cost`, `budget_annual`, `gpa_normalized` (0-1), `profile_strength_score` (0-100), `work_experience_years`. All costs/budgets must share one annual currency, and all predictors must have been available before the independently reviewed outcome at a documented decision horizon. Labels must not be generated from the arithmetic function or candidate model. Do not supply final award amounts, award decisions, or post-outcome funding as predictors. Verify provenance before setting `LABELS_ARE_REAL=True`; a filename alone does not establish genuine outcomes.

The fixed split is approximately 60% training students, 20% validation students, 20% held-out test students. No student appears in multiple partitions. Pipelines learn imputers/scalers on training rows only. A majority baseline, logistic regression, and histogram gradient boosting compete on validation balanced accuracy, then macro F1. Each validation/test student contributes one row chosen by lexical `record_id` before looking at labels; repeated records cannot inflate sample size. The winner is refit on training plus validation students and evaluated once on untouched test students. Do not rerun this test to select new features/hyperparameters; reserve a new holdout for subsequent iterations.

Report accuracy, balanced accuracy, macro F1, confusion matrix, and 95% Wilson interval for accuracy. A wide interval or few independent students limits evidence even when a point estimate exceeds 96%. The optional model predicts independently reviewed support need, not deterministic funding-gap arithmetic or scholarship eligibility.

References: [scikit-learn leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html) and [group-separated splitting](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html).


In [5]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

AGENT5_FEATURES = ['total_cost', 'budget_annual', 'gpa_normalized', 'profile_strength_score', 'work_experience_years']

def wilson_interval(correct, count, z=1.959963984540054):
    p, denominator = correct / count, 1 + z * z / count
    centre = (p + z * z / (2 * count)) / denominator
    radius = z * math.sqrt(p * (1 - p) / count + z * z / (4 * count * count)) / denominator
    return [max(0.0, centre - radius), min(1.0, centre + radius)]

def evaluate_agent5_real_labels(frame, *, labels_are_real=False):
    if not labels_are_real:
        raise ValueError('Verify independent real labels and provenance, then set labels_are_real=True.')
    required = {'record_id', 'student_id', 'label_source', 'needs_financial_support', *AGENT5_FEATURES}
    if required - set(frame.columns):
        raise ValueError('Labelled data missing columns: ' + ', '.join(sorted(required - set(frame.columns))))
    data = frame.copy()
    for column in ('record_id', 'student_id', 'label_source'):
        data[column] = data[column].map(clean)
        if data[column].eq('').any():
            raise ValueError(column + ' must be nonblank.')
    if data['record_id'].duplicated().any():
        raise ValueError('record_id must be unique; remove duplicated records before evaluation.')
    if data['student_id'].nunique() < 30:
        raise ValueError('At least 30 independent students are required; substantially more improve precision.')
    labels = pd.to_numeric(data['needs_financial_support'], errors='raise')
    if labels.isna().any() or set(labels.unique()) != {0, 1}:
        raise ValueError('needs_financial_support must contain both binary classes 0 and 1.')
    X = data[AGENT5_FEATURES].apply(pd.to_numeric, errors='raise').astype(float)
    if np.isinf(X.to_numpy()).any() or X.notna().sum().eq(0).any():
        raise ValueError('Infinite or entirely missing feature columns cannot be evaluated.')
    if (X.lt(0).any().any() or X['gpa_normalized'].gt(1).any()
            or X['profile_strength_score'].gt(100).any()):
        raise ValueError('Feature ranges violate the documented input schema.')
    y = labels.astype(int)
    development, test = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42).split(X, y, data['student_id']))
    inner_train, inner_val = next(GroupShuffleSplit(n_splits=1, test_size=.25, random_state=43).split(
        X.iloc[development], y.iloc[development], data['student_id'].iloc[development]))
    train, val = development[inner_train], development[inner_val]
    def independent_rows(indices):
        return data.iloc[indices].assign(_position=indices).sort_values('record_id').drop_duplicates('student_id')['_position'].to_numpy()
    val, test = independent_rows(val), independent_rows(test)
    for name, indices in [('train', train), ('validation', val), ('test', test)]:
        if y.iloc[indices].nunique() != 2:
            raise ValueError(name + ' lacks a class with the fixed group split; collect more labelled students.')
    sets = [set(data['student_id'].iloc[part]) for part in (train, val, test)]
    assert all(not sets[i] & sets[j] for i in range(3) for j in range(i + 1, 3))
    models = {
        'majority_baseline': make_pipeline(SimpleImputer(strategy='median', add_indicator=True), DummyClassifier(strategy='most_frequent')),
        'logistic': make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
        'hist_gradient_boosting': make_pipeline(SimpleImputer(strategy='median', add_indicator=True), HistGradientBoostingClassifier(max_iter=150, max_leaf_nodes=15, l2_regularization=2, random_state=42)),
    }
    validation = []
    for name, model in models.items():
        model.fit(X.iloc[train], y.iloc[train])
        prediction = model.predict(X.iloc[val])
        validation.append({'model': name, 'accuracy': accuracy_score(y.iloc[val], prediction),
            'balanced_accuracy': balanced_accuracy_score(y.iloc[val], prediction),
            'macro_f1': f1_score(y.iloc[val], prediction, average='macro', zero_division=0)})
    table = pd.DataFrame(validation).sort_values(['balanced_accuracy', 'macro_f1', 'model'], ascending=[False, False, True])
    winner = str(table.iloc[0]['model'])
    fitted = clone(models[winner]).fit(X.iloc[development], y.iloc[development])
    prediction = fitted.predict(X.iloc[test])
    correct, count = int((prediction == y.iloc[test].to_numpy()).sum()), len(test)
    report = {'selected_model': winner, 'evaluation_unit': 'one prespecified record per held-out student',
        'training_students': len(sets[0]), 'validation_students': len(sets[1]), 'test_students': count,
        'accuracy': accuracy_score(y.iloc[test], prediction),
        'balanced_accuracy': balanced_accuracy_score(y.iloc[test], prediction),
        'macro_f1': f1_score(y.iloc[test], prediction, average='macro', zero_division=0),
        'confusion_matrix_labels': [0, 1], 'confusion_matrix': confusion_matrix(y.iloc[test], prediction, labels=[0, 1]).tolist(),
        'accuracy_wilson_95': wilson_interval(correct, count),
        'minimum_90_percent_reached': correct / count >= .90,
        'target_96_percent_reached': correct / count >= TARGET_ACCURACY,
        'target_supported_by_interval': wilson_interval(correct, count)[0] >= TARGET_ACCURACY}
    return {'model': fitted, 'validation_results': table, 'test_report': report,
        'feature_columns': AGENT5_FEATURES.copy(), 'test_record_ids': data['record_id'].iloc[test].tolist()}


agent5_evaluation = None
print("Funding-support evaluation is an explicit separate action, never run automatically.")
print("Requires independent outcomes, >=30 students, and labels_are_real=True.")

Funding-support evaluation is an explicit separate action, never run automatically.
Requires independent outcomes, >=30 students, and labels_are_real=True.


## 5. Correctness checks only - explicitly fictional fixtures

These local examples verify arithmetic and boundary behavior. They are fictional, never used for training or accuracy claims, and not real scholarship recommendations. They test expired deadlines, unknown nationality, GPA scales, currency mismatch, annual costs/budgets, duplicate confirmed awards, nonstackable awards, mutually exclusive groups, degree restrictions, Agent 7 merit, input-state preservation, and missing catalogues.


In [6]:
from copy import deepcopy

fixture_catalog = pd.DataFrame([{
    'scholarship_id': 'FICTIONAL-001', 'scholarship_name': 'Fictional fixture - not an actual scholarship',
    'currency': 'USD', 'coverage_basis': 'tuition', 'amount_period': 'annual',
    'coverage_percent': 50, 'amount': np.nan, 'eligible_nationalities': 'India',
    'eligible_fields': 'Computer Science', 'eligible_degrees': 'masters',
    'min_gpa': 3.2, 'gpa_scale': 4, 'deadline_policy': 'fixed', 'deadline': '2027-01-01',
    'description': 'Fictional computer science fixture', 'university': 'Fixture University'}])
fixture_state = {'profile': {'nationality': 'India', 'field_of_study': 'Computer Science',
    'target_degree': 'MS', 'gpa': 8.5, 'gpa_scale': 10, 'budget_annual': 10000, 'budget_currency': 'USD'},
    'finance': [{'university': 'Fixture University', 'program_name': 'MS Computer Science',
        'total_cost': 30000, 'tuition_annual': 20000, 'currency': 'USD'}]}
fixture_original = deepcopy(fixture_state)
fixture_result = scholarship_agent_v2(fixture_state, fixture_catalog, as_of='2026-10-20')['scholarships'][0]
assert fixture_result['funding_gap'] == 20000  # Potential award does not reduce the gap.
assert fixture_result['recommendations'][0]['potential_annual_coverage'] == 10000
assert fixture_state == fixture_original
assert scholarship_agent_v2(fixture_state, None, as_of='2026-10-20')['scholarships'][0]['recommendations'] == []
assert scholarship_agent_v2(fixture_state, fixture_catalog, as_of='2027-01-02')['scholarships'][0]['recommendations'] == []
fixture_unknown = deepcopy(fixture_state)
fixture_unknown['profile'].pop('nationality')
assert scholarship_agent_v2(fixture_unknown, fixture_catalog, as_of='2026-10-20')['scholarships'][0]['eligibility_review'][0]['eligibility_match'] == 'unknown'
fixture_low_gpa = deepcopy(fixture_state)
fixture_low_gpa['profile']['gpa'] = 7.0
assert scholarship_agent_v2(fixture_low_gpa, fixture_catalog, as_of='2026-10-20')['scholarships'][0]['recommendations'] == []
fixture_currency = fixture_catalog.copy()
fixture_currency['currency'] = 'EUR'
assert scholarship_agent_v2(fixture_state, fixture_currency, as_of='2026-10-20')['scholarships'][0]['recommendations'] == []
fixture_degree = fixture_catalog.copy()
fixture_degree['eligible_degrees'] = 'MBA'
assert scholarship_agent_v2(fixture_state, fixture_degree, as_of='2026-10-20')['scholarships'][0]['recommendations'] == []
fixture_merit = fixture_catalog.copy()
fixture_merit['min_profile_strength_score'] = 75
fixture_with_merit = {**fixture_state, 'extracurricular': {'profile_strength_score': 82}}
assert scholarship_agent_v2(fixture_with_merit, fixture_merit, as_of='2026-10-20')['scholarships'][0]['recommendations']
assert scholarship_agent_v2(fixture_state, fixture_merit, as_of='2026-10-20')['scholarships'][0]['eligibility_review'][0]['eligibility_match'] == 'unknown'
for fixture_field, fixture_value in [('budget_annual', None), ('budget_annual', -1), ('budget_currency', 'INR')]:
    fixture_bad = deepcopy(fixture_state)
    fixture_bad['profile'][fixture_field] = fixture_value
    assert scholarship_agent_v2(fixture_bad, as_of='2026-10-20')['scholarships'][0]['funding_gap'] is None
for fixture_field, fixture_value in [('total_cost', None), ('total_cost', -1), ('cost_period', 'monthly'), ('tuition_annual', 40000)]:
    fixture_bad = deepcopy(fixture_state)
    fixture_bad['finance'][0][fixture_field] = fixture_value
    assert scholarship_agent_v2(fixture_bad, as_of='2026-10-20')['scholarships'][0]['funding_gap'] is None
fixture_award = {'scholarship_id': 'CONFIRMED-FIXTURE-A', 'status': 'confirmed',
    'university': 'Fixture University', 'program_name': 'MS Computer Science', 'currency': 'USD',
    'amount_period': 'annual', 'annual_amount': 6000, 'stackable': True, 'exclusive_group': 'fixture-group'}
fixture_awards = [fixture_award, dict(fixture_award),
    {**fixture_award, 'scholarship_id': 'CONFIRMED-FIXTURE-B', 'annual_amount': 8000},
    {**fixture_award, 'scholarship_id': 'CONFIRMED-FIXTURE-C', 'annual_amount': 9000, 'stackable': False}]
assert confirmed_funding(fixture_awards, fixture_state['finance'][0])[0] == 9000
fixture_confirmed = {**fixture_state, 'confirmed_awards': fixture_awards}
assert scholarship_agent_v2(fixture_confirmed, fixture_catalog, as_of='2026-10-20')['scholarships'][0]['funding_gap'] == 11000
assert confirmed_funding([fixture_award, {**fixture_award, 'annual_amount': 7000}], fixture_state['finance'][0])[0] == 0
assert confirmed_funding([fixture_award, {**fixture_award, 'amount_period': 'monthly'}], fixture_state['finance'][0])[0] == 0
assert confirmed_funding([{**fixture_award, 'status': 'applied'}], fixture_state['finance'][0])[0] == 0
fixture_ms_only = fixture_catalog.copy()
fixture_ms_only['eligible_degrees'] = 'MS'
fixture_mba = deepcopy(fixture_state)
fixture_mba['profile']['target_degree'] = 'MBA'
assert scholarship_agent_v2(fixture_mba, fixture_ms_only, as_of='2026-10-20')['scholarships'][0]['recommendations'] == []
assert scholarship_agent_v2(fixture_mba, fixture_catalog, as_of='2026-10-20')['scholarships'][0]['recommendations']
fixture_upstream = deepcopy(fixture_state)
fixture_upstream['profile']['target_program'] = fixture_upstream['profile'].pop('field_of_study')
fixture_upstream['profile']['interests_text'] = 'Computer Science'
fixture_upstream['profile'].pop('budget_annual')
fixture_upstream['profile'].pop('budget_currency')
fixture_upstream['preferences'] = {'budget_max_usd': 10000}
assert scholarship_agent_v2(fixture_upstream, fixture_catalog, as_of='2026-10-20')['scholarships'][0]['funding_gap'] is None
fixture_upstream['preferences']['budget_period'] = 'annual'
assert scholarship_agent_v2(fixture_upstream, fixture_catalog, as_of='2026-10-20')['scholarships'][0]['funding_gap'] == 20000
fixture_bad_date = fixture_catalog.copy()
fixture_bad_date['deadline'] = '2027'
assert scholarship_agent_v2(fixture_state, fixture_bad_date, as_of='2026-10-20')['scholarships'][0]['eligibility_review'][0]['eligibility_match'] == 'unknown'
print('Fictional-fixture correctness checks passed. No training or predictive-accuracy benchmark was run.')
print('Fictional example: cost USD 30,000, budget USD 10,000, potential award USD 10,000.')
print('Funding gap before any confirmed award:', fixture_result['funding_gap'])
print('Real scholarship catalogue:', 'available' if scholarship_catalog is not None else 'not supplied')
print('Measured predictive accuracy:', agent5_evaluation['test_report']['accuracy'] if agent5_evaluation else 'NOT MEASURED')
print('Desired at least 96% accuracy: NOT MEASURED without real labelled outcomes.')


alias_profile = dict(fixture_state['profile'], nationality='Indian')
alias_finance = fixture_state['finance'][0]
assert eligible(fixture_catalog.iloc[0].to_dict(), alias_profile, alias_finance,
                pd.Timestamp('2026-10-20').date())[0] == 'eligible'
alias_profile['field_of_study'] = 'Computer Science and Engineering'
assert eligible(fixture_catalog.iloc[0].to_dict(), alias_profile, alias_finance,
                pd.Timestamp('2026-10-20').date())[0] == 'unknown'
alias_profile.update(nationality='Unmapped demonym', field_of_study='Computer Science')
assert eligible(fixture_catalog.iloc[0].to_dict(), alias_profile, alias_finance,
                pd.Timestamp('2026-10-20').date())[0] == 'unknown'
alias_profile['nationality'] = 'Canada'
assert eligible(fixture_catalog.iloc[0].to_dict(), alias_profile, alias_finance,
                pd.Timestamp('2026-10-20').date())[0] == 'ineligible'
print('Alias checks passed: Indian/India; related fields and unmapped demonyms require review; explicit country exclusions remain enforced.')

Fictional-fixture correctness checks passed. No training or predictive-accuracy benchmark was run.
Fictional example: cost USD 30,000, budget USD 10,000, potential award USD 10,000.
Funding gap before any confirmed award: 20000.0
Real scholarship catalogue: available
Measured predictive accuracy: NOT MEASURED
Desired at least 96% accuracy: NOT MEASURED without real labelled outcomes.
Alias checks passed: Indian/India; related fields and unmapped demonyms require review; explicit country exclusions remain enforced.


## 6. Use with your real pipeline

After running Agent 4, call:

```python
# state already contains your real profile and finance entries.
# Optionally pass a sentence-transformer instance already loaded by Agent 3.
updated_state = scholarship_agent_v2(
    state, scholarship_catalog, as_of="2026-10-20", embed_model=None
)
updated_state["scholarships"]
```

Supply the actual assessment date when calling the example. A real catalogue and held-out, independently labelled student outcomes are still required before claiming 96% accuracy. A predicted label can support review; it cannot establish that a scholarship is awarded. The optional classifier is returned by `evaluate_agent5_real_labels`, together with feature order, validation results, and test report. It is a development utility, not silently connected to the deterministic funding outputs.


## 7. Demonstration using your actual catalog

The student and cost estimate below are illustrative; scholarship records are
loaded from your files. This is an integration demonstration, not an outcome test.
No generated student labels are used for model training. The budget is explicitly
annual and denominated in USD, following the existing shared-state format.

For your real pipeline use `scholarship_agent(state, scholarship_catalog)`.
Pass the actual assessment date or omit `as_of` to use today's date in Asia/Kolkata.

In [7]:
# Match the naming convention of program_agent / financial_agent in existing notebooks.
scholarship_agent = scholarship_agent_v2
agent5_demo_state = {
    'student_id': 'illustrative-profile-not-an-observed-outcome',
    'profile': {'nationality': 'India', 'target_program': 'Computer Science',
                'target_degree': 'MS', 'gpa': 8.6, 'gpa_scale': 10.0,
                'interests_text': 'computer science, distributed systems and software engineering',
                'work_experience_years': 2},
    'preferences': {'budget_max_usd': 20000, 'budget_period': 'annual'},
    'extracurricular': {'profile_strength_score': 82},
    'matched_universities': [{'university': 'Stanford University', 'country': 'USA'}],
    'finance': [{'university': 'Stanford University', 'program_name': 'MS Computer Science',
                 'tuition_annual': 35000, 'living_cost_annual': 15000, 'total_cost': 50000,
                 'currency': 'USD', 'cost_data_quality': 'ILLUSTRATIVE_EXAMPLE'}],
}
if scholarship_catalog is not None:
    agent5_catalog_demo = scholarship_agent(agent5_demo_state, scholarship_catalog, as_of='2026-10-20')
    demo_finance = agent5_catalog_demo['scholarships'][0]
    print('Illustrative annual funding gap (USD):', demo_finance['funding_gap'])
    print('Full-catalog eligibility counts:', demo_finance['catalog_counts'])
    print('Fully checked recommendations returned:', len(demo_finance['recommendations']))
    print('Ranked catalog records requiring review:', len(demo_finance['review_candidates']))
    for record in demo_finance['review_candidates'][:5]:
        print(record['scholarship_id'], '|', record['scholarship_name'])
        print('  Eligibility:', record['eligibility_match'], '| Review:', '; '.join(record['reasons']))
    assert demo_finance['funding_gap'] == 30000
    assert not set(item['scholarship_id'] for item in demo_finance['review_candidates']) - set(scholarship_catalog['scholarship_id'])
    assert 'scholarships' not in agent5_demo_state

# Parser checks are functional tests, not measured scholarship accuracy.
assert explicit_date('31 October 2026') == '2026-10-31'
assert explicit_date('Program-dependent (typically Jun-Oct 2026)') == ''
assert pd.isna(explicit_gpa_scale({}, 'Minimum GPA 3.5'))
assert explicit_gpa_scale({}, 'Minimum GPA 3.5 on a 4.0 scale') == 4.0
assert explicit_award_period({'amount': 5000}, 'Monthly stipend and tuition support') == ''
assert explicit_award_period({'amount': 5000}, 'Grant of USD 5,000 per year') == 'annual'
assert explicit_award_period({'amount': 5000}, 'Total course cost USD 15,000 per year') == ''
assert degree('MS') != degree('MBA')
print('Uploaded-data adapter and real-catalog integration checks passed.')
print('Target >=96%: NOT VERIFIED; independent student relevance/outcome labels are not in these catalog files.')

Illustrative annual funding gap (USD): 30000.0
Full-catalog eligibility counts: {'examined': 9673, 'fully_recommendable': 0, 'needs_review': 630, 'ineligible': 9043}
Fully checked recommendations returned: 0
Ranked catalog records requiring review: 10
SCH009659 | Ambassador Apartments Scholarship in Computer Science
  Eligibility: unknown | Review: unknown: university; unknown: GPA or GPA scale; unknown: deadline policy; unknown: source academic/English/other requirements need student-specific review; nonannual or missing award period
SCH008630 | Information Technology Scholarship
  Eligibility: unknown | Review: unknown: university; unknown: GPA or GPA scale; unknown: deadline policy; unknown: source academic/English/other requirements need student-specific review; nonannual or missing award period
SCH005661 | LITA/Christian Larew Memorial Scholarship
  Eligibility: unknown | Review: unknown: university; unknown: GPA or GPA scale; unknown: deadline policy; unknown: source academic/Eng

## 8. Upstream budget and country contract checks

The actual Agent 2 and Agent 4 notebook functions are exercised with fixture
retrieval/model/cost dependencies. These checks do not contact external services,
train a model, or establish accuracy. Agent 2 now emits the annual budget period;
Agent 4 carries the destination country into `finance`.

In [8]:
import ast as _handoff_ast
import json as _handoff_json
from types import SimpleNamespace as _FixtureCandidate

_project = next(root for root in (Path.cwd(), *Path.cwd().parents) if (root / "data/raw").is_dir())
def _notebook_function_source(relative, function_name):
    document = _handoff_json.loads((_project / relative).read_text(encoding="utf-8"))
    return next("".join(c["source"]) for c in document["cells"]
                if c["cell_type"] == "code" and f"def {function_name}(" in "".join(c["source"]))

_a2_source = _notebook_function_source(
    "backend/app/agents/university_matching_agent/AGENT2_University_agent.ipynb", "university_matching_agent")
_a2_scope = {
    "Agent7ScoreMissingError": ValueError,
    "retrieve_candidates": lambda *args, **kwargs: [_FixtureCandidate(payload={
        "university_name": "Fixture University", "program_name": "MS Computer Science",
        "country": "USA"})],
    "rerank": lambda query, candidates, **kwargs: [(candidate, .8) for candidate in candidates],
    "compute_final_score": lambda *args: .8,
    "compute_tier_ml": lambda *args: ("Target", .6, [], 3.4, 80),
    "generate_explanation": lambda *args, **kwargs: "CONTRACT_TEST_FIXTURE",
    "tag_budget_signal": lambda *args: None,
    "save_matches_to_db": lambda *args: None,
}
exec(compile(_handoff_ast.parse(_a2_source), "agent2-contract", "exec"), _a2_scope)
_budget_state = {"profile": {"gpa": 8.5, "gpa_scale": 10, "target_degree": "MS",
                            "target_program": "Computer Science", "nationality": "India"},
                 "preferences": {"budget_max_usd": 10000},
                 "extracurricular": {"profile_strength_score": 72}}
_a2_result = _a2_scope["university_matching_agent"](_budget_state)
assert _a2_result["preferences"]["budget_period"] == "annual"
assert _a2_result["preferences"]["budget_currency"] == "USD"
assert _a2_result["matched_universities"][0]["country"] == "USA"
_a2_result["programs"] = [{"university": "Fixture University", "program_name": "MS Computer Science",
                          "country": "USA", "admit_probability": None, "tier": "Unknown"}]
_a4_source = _notebook_function_source(
    "backend/app/agents/financial_agent/Agent4_Financial_agent.ipynb", "financial_agent")
_a4_scope = {"estimate_cost": lambda *args: {
    "total_cost": 30000, "tuition_annual": 20000, "living_cost_annual": 10000,
    "currency": "USD", "cost_data_quality": "CONTRACT_TEST_FIXTURE"}}
exec(compile(_handoff_ast.parse(_a4_source), "agent4-contract", "exec"), _a4_scope)
_finance_state = _a4_scope["financial_agent"](_a2_result)
assert _finance_state["finance"][0]["country"] == "USA"
_scholarship_state = scholarship_agent(_finance_state, fixture_catalog, as_of="2026-10-20")
assert _scholarship_state["scholarships"][0]["funding_gap"] == 20000
_monthly_state = {"preferences": {"budget_max_usd": 1000, "budget_period": "monthly"}}
assert _a2_scope["normalize_agent2_budget_preferences"](_monthly_state)["budget_period"] == "monthly"
_monthly_finance = {**_finance_state, "preferences": _monthly_state["preferences"]}
assert scholarship_agent(_monthly_finance, None, as_of="2026-10-20")["scholarships"][0]["funding_gap"] is None
agent5_contract_status = "PASSED: legacy annual budget, 0-100 merit, country propagation and explicit nonannual budgets."
print(agent5_contract_status)

PASSED: legacy annual budget, 0-100 merit, country propagation and explicit nonannual budgets.


## Independent student–scholarship eligibility evaluation

Use `agent5_eligibility_judgments.csv`, the shared real profile file, and
`agent5_evaluation_states.json` exported from actual Agent 4 contexts.
Human `checked_on` determines the agent's `as_of` date. A source URL is provenance to
verify manually; its presence alone does not prove an official review occurred.

Test requires at least 200 pairs, with Tier A and non-Tier A records. Report selective
accuracy **and** decision coverage/unknown share. Human `cannot_determine` pairs cannot
establish correctness of a definitive decision, so report them separately.
All-unknown output has undefined accuracy and zero coverage.
Pairs sharing a student remain in all metrics, but their ordinary Wilson interval is
descriptive and cannot support an independent-sample 96% claim.

Source enrichments contain only explicit periods, scales and coverage bases, with evidence.
They do not waive English/academic requirements or turn an uncertain scholarship into a
confirmed award. Run All never runs the eligibility test.

In [9]:
import hashlib as _ev_hashlib
import json as _ev_json
from pathlib import Path as _EvPath
from datetime import datetime as _EvDatetime, timezone as _EvTimezone
import pandas as _ev_pd
import numpy as _ev_np

EVALUATION_ROOT = next(p for p in (_EvPath.cwd(), *_EvPath.cwd().parents)
                       if (p / "data/raw").is_dir())
PROFILE_COLUMNS = "student_id query_group interests_text target_program target_degree gpa gpa_scale nationality budget_annual budget_currency profile_strength_score work_experience_years split".split()

def load_research_profiles(path=None):
    path = path or EVALUATION_ROOT / "data/raw/student_profiles/student_profiles_anonymized.csv"
    frame = _ev_pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = set(PROFILE_COLUMNS) - set(frame)
    if missing:
        raise ValueError(f"Profile columns missing: {sorted(missing)}")
    if frame.empty:
        raise ValueError("Collect consented, anonymized real profiles; the template has no rows.")
    for name in PROFILE_COLUMNS:
        frame[name] = frame[name].str.strip()
        if frame[name].eq("").any():
            raise ValueError(f"Blank profile field: {name}; do not invent missing values.")
    if frame.student_id.duplicated().any():
        raise ValueError("One profile per student_id; put query paraphrases in the annotation packet.")
    if not frame.split.isin(["dev", "test"]).all():
        raise ValueError("Assign dev/test before model output is viewed.")
    if frame.groupby("query_group").split.nunique().gt(1).any():
        raise ValueError("A student/query group crosses splits.")
    if frame.query_group.duplicated().any():
        raise ValueError("Each query_group must identify one student in the shared profile file.")
    for name in ["gpa", "gpa_scale", "budget_annual", "profile_strength_score", "work_experience_years"]:
        frame[name] = _ev_pd.to_numeric(frame[name], errors="raise")
        if not _ev_np.isfinite(frame[name]).all() or frame[name].lt(0).any():
            raise ValueError(f"Invalid nonnegative numeric profile field: {name}")
    if (frame.gpa_scale.le(0).any() or frame.gpa.gt(frame.gpa_scale).any()
            or frame.profile_strength_score.gt(100).any()):
        raise ValueError("GPA must fit its explicit scale; merit score uses 0-100.")
    if not frame.budget_currency.str.fullmatch(r"[A-Z]{3}").all():
        raise ValueError("budget_currency must be an explicit three-letter uppercase currency.")
    return frame

def research_profile_state(row):
    profile = {k: row[k] for k in PROFILE_COLUMNS if k not in ("student_id", "query_group", "split")}
    preferences = {"budget_period": "annual", "budget_currency": row["budget_currency"]}
    if row["budget_currency"] == "USD":
        preferences["budget_max_usd"] = float(row["budget_annual"])
    return {"student_id": row["student_id"], "profile": profile, "preferences": preferences,
            "extracurricular": {"profile_strength_score": float(row["profile_strength_score"])}}

def _ev_sha(path):
    return _ev_hashlib.sha256(_EvPath(path).read_bytes()).hexdigest()

def _ev_write_new(path, value):
    path = _EvPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8") as handle:
        _ev_json.dump(value, handle, indent=2, allow_nan=False)

def _ev_now():
    return _EvDatetime.now(_EvTimezone.utc).isoformat()

A5_DATA = EVALUATION_ROOT / "data/raw/agent5"
A5_RUNS = A5_DATA / "eligibility_evaluation_runs"
ELIGIBILITY_COLUMNS = "pair_id student_id split scholarship_id human_decision source_url checked_on annotator_1 annotator_2 notes".split()

def export_agent5_evaluation_states(pair_states, output_path=None):
    """Save actual pipeline states keyed by pair_id, before human eligibility labels are supplied.

    A pair must specify one actual Agent 4 finance entry, including destination country.
    No default university, costs, or scholarship host country is substituted.
    """
    if not pair_states:
        raise ValueError("Supply real pipeline states keyed by planned pair_id.")
    for pair_id, state in pair_states.items():
        if len(state.get("finance", [])) != 1 or not state.get("student_id"):
            raise ValueError(f"{pair_id}: select one real finance context and student_id.")
        finance = state["finance"][0]
        if any(not clean(finance.get(key)) for key in ("university", "program_name", "country", "currency")):
            raise ValueError(f"{pair_id}: university, program, country and currency are required.")
        if clean(finance.get("cost_period")).casefold() != "annual" or not compatible_cost(finance):
            raise ValueError(f"{pair_id}: annual compatible Agent 4 cost data is required.")
    destination = output_path or A5_DATA / "agent5_evaluation_states.json"
    _ev_write_new(destination, {"created_at": _ev_now(), "pair_states": pair_states})
    print("Real evaluation contexts saved. Keep this private along with profiles and judgments.")

def apply_agent5_source_enrichments(catalog, path=None):
    """Apply only explicit, traceable units; never infer periods/scales or waive other rules."""
    path = path or A5_DATA / "agent5_source_enrichments.csv"
    if not _EvPath(path).exists():
        return catalog.copy()
    updates = _ev_pd.read_csv(path, dtype=str, keep_default_na=False)
    output = catalog.copy().set_index("scholarship_id")
    allowed = {"gpa_scale", "amount_period", "coverage_basis"}
    required = {"scholarship_id", "field", "value", "source_url", "checked_on", "reviewer_id", "evidence"}
    if required - set(updates):
        raise ValueError("Source enrichment file has missing columns.")
    if updates.duplicated(["scholarship_id", "field"]).any():
        raise ValueError("Conflicting duplicate scholarship/field enrichments.")
    for row in updates.to_dict("records"):
        if any(not row[k].strip() for k in required) or row["field"] not in allowed:
            raise ValueError("Every enrichment needs an explicit value, source evidence, reviewer and date.")
        if row["scholarship_id"] not in output.index or not row["source_url"].startswith("https://"):
            raise ValueError("Unknown scholarship or invalid official source URL.")
        _ev_pd.Timestamp(row["checked_on"])
        value = row["value"]
        if row["field"] == "gpa_scale":
            value = float(value)
            if not _ev_np.isfinite(value) or value <= 0:
                raise ValueError("GPA scale must be explicit and positive.")
        elif row["field"] == "amount_period" and value not in ("annual", "one_time", "monthly", "semester"):
            raise ValueError("Use an explicit supported period. Nonannual awards remain unconfirmed for annual gap.")
        elif row["field"] == "coverage_basis" and value not in ("tuition", "total"):
            raise ValueError("Coverage basis must explicitly be tuition or total.")
        output.at[row["scholarship_id"], row["field"]] = value
    return validate_catalog(output.reset_index())

def summarize_agent5_pair_predictions(predictions):
    """Selective accuracy needs BOTH an agent decision and a resolvable human label."""
    if predictions.empty:
        raise ValueError("No evaluated pairs.")
    decided = predictions.agent_decision.isin(["eligible", "ineligible"])
    resolved = predictions.human_decision.isin(["eligible", "ineligible"])
    scored = decided & resolved
    correct = int(predictions.loc[scored, "agent_decision"].eq(predictions.loc[scored, "human_decision"]).sum())
    count = int(scored.sum())
    independent = predictions.loc[scored, "student_id"].nunique() == count
    interval = wilson_interval(correct, count) if count else None
    return {"pairs": len(predictions), "students": int(predictions.student_id.nunique()),
        "agent_decisions": int(decided.sum()), "unknown_count": int((~decided).sum()),
        "decision_coverage": float(decided.mean()), "unknown_share": float((~decided).mean()),
        "human_resolved_pairs": int(resolved.sum()), "scored_decisions": count,
        "decisions_on_human_cannot_determine": int((decided & ~resolved).sum()),
        "correct_decisions": correct, "selective_accuracy": correct / count if count else None,
        "decided_and_human_resolved_share": float(scored.mean()),
        "accuracy_wilson_95": interval,
        "wilson_independent_students": bool(independent),
        "selective_96_supported": bool(count and independent and interval[0] >= .96),
        "fully_recommended_pairs": int(predictions.fully_recommended.sum()),
        "three_way_agreement": float(predictions.agent_decision.replace({"unknown": "cannot_determine"})
                                     .eq(predictions.human_decision).mean())}

def _evaluate_agent5_pairs(judgments, profiles, contexts, catalog, split, minimum_pairs):
    missing = set(ELIGIBILITY_COLUMNS) - set(judgments)
    if missing:
        raise ValueError(f"Eligibility judgment columns missing: {sorted(missing)}")
    all_rows = judgments.copy()
    for name in ELIGIBILITY_COLUMNS:
        all_rows[name] = all_rows[name].astype(str).str.strip()
        if name != "notes" and all_rows[name].eq("").any():
            raise ValueError(f"Blank judgment field: {name}")
    if (all_rows.pair_id.duplicated().any()
            or all_rows.duplicated(["student_id", "scholarship_id", "checked_on"]).any()):
        raise ValueError("Duplicate pair ID or repeated assessment of the same student/scholarship/date.")
    if (not all_rows.split.isin(["dev", "test"]).all()
            or all_rows.groupby("student_id").split.nunique().gt(1).any()):
        raise ValueError("Invalid split or a student crosses dev/test.")
    if not all_rows.human_decision.isin(["eligible", "ineligible", "cannot_determine"]).all():
        raise ValueError("human_decision must be eligible, ineligible or cannot_determine.")
    rows = all_rows.loc[all_rows.split.eq(split)].copy()
    if len(rows) < minimum_pairs:
        raise ValueError(f"Need at least {minimum_pairs} {split} pairs; found {len(rows)}.")
    if not rows.source_url.str.startswith("https://").all():
        raise ValueError("An official HTTPS source page is required for every judgment.")
    if (rows.annotator_1 == rows.annotator_2).any():
        raise ValueError("Use two distinct human annotator IDs; notes retain adjudication details.")
    profiles = profiles.set_index("student_id")
    by_id = validate_catalog(catalog).set_index("scholarship_id", drop=False)
    strata = rows.scholarship_id.isin(preferred_scholarship_ids)
    if split == "test" and strata.nunique() != 2:
        raise ValueError("Test pairs must include Tier A AND the rest of the catalog.")
    predictions = []
    for row in rows.to_dict("records"):
        if row["student_id"] not in profiles.index or row["scholarship_id"] not in by_id.index:
            raise ValueError("Unknown student or scholarship ID; no rows are silently dropped.")
        profile = profiles.loc[row["student_id"]].to_dict()
        profile["student_id"] = row["student_id"]
        if profile["split"] != split:
            raise ValueError("Judgment split disagrees with the fixed student split.")
        context = contexts.get(row["pair_id"])
        if not context or context.get("student_id") != row["student_id"] or len(context.get("finance", [])) != 1:
            raise ValueError("Each pair needs its actual one-program Agent 4 context snapshot.")
        finance = context["finance"][0]
        if (clean(finance.get("cost_period")).casefold() != "annual" or not compatible_cost(finance)
                or any(not clean(finance.get(k)) for k in ("university", "program_name", "country"))):
            raise ValueError("Incomplete annual cost or destination context.")
        date = pd.Timestamp(row["checked_on"])
        if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", row["checked_on"]) or pd.isna(date) or date.date() > pd.Timestamp.now(tz="Asia/Kolkata").date():
            raise ValueError("checked_on must be a real ISO review date, not a future date.")
        state = {**context, **research_profile_state(profile), "finance": context["finance"]}
        candidate = by_id.loc[[row["scholarship_id"]]].reset_index(drop=True)
        # Invoke the public agent on exactly the independently judged pair.
        result = scholarship_agent(state, catalog=candidate, as_of=row["checked_on"], top_k=1)["scholarships"][0]
        decisions = result["recommendations"] + result["eligibility_review"]
        if len(decisions) != 1:
            raise AssertionError("Every input pair must have exactly one explicit eligibility decision.")
        predictions.append({**row, "agent_decision": decisions[0]["eligibility_match"],
            "fully_recommended": bool(result["recommendations"]),
            "tier_a": row["scholarship_id"] in preferred_scholarship_ids,
            "reasons": decisions[0].get("reasons", [])})
    predictions = pd.DataFrame(predictions)
    report = summarize_agent5_pair_predictions(predictions)
    report["split"] = split
    report["by_tier"] = {str(tier): summarize_agent5_pair_predictions(group)
                          for tier, group in predictions.groupby("tier_a")}
    report["confusion_matrix"] = pd.crosstab(predictions.human_decision, predictions.agent_decision).to_dict()
    return {"report": report, "predictions": predictions}

def _agent5_evaluation_inputs():
    profiles = load_research_profiles()
    judgments = pd.read_csv(A5_DATA / "agent5_eligibility_judgments.csv", dtype=str, keep_default_na=False)
    contexts = _ev_json.loads((A5_DATA / "agent5_evaluation_states.json").read_text(encoding="utf-8"))["pair_states"]
    catalog = apply_agent5_source_enrichments(scholarship_catalog)
    return judgments, profiles, contexts, catalog

def evaluate_agent5_eligibility_dev():
    """Review dev failures only. This is not a held-out accuracy claim."""
    judgments, profiles, contexts, catalog = _agent5_evaluation_inputs()
    return _evaluate_agent5_pairs(judgments, profiles, contexts, catalog, "dev", 1)

def _a5_fingerprint():
    notebook = EVALUATION_ROOT / "backend/app/agents/scholarship_agent/Agent5_Scholarship_agent.ipynb"
    source = _ev_json.loads(notebook.read_text(encoding="utf-8"))
    digest = _ev_hashlib.sha256("".join("".join(c["source"]) for c in source["cells"]
                                      if c["cell_type"] == "code").encode())
    # Persisted source plus the active functions: editing a live function also changes the hash.
    import marshal
    for function in (scholarship_agent, eligible, potential_coverage, country_name, normalize_field,
                     degree, tokens, clean, number, intended_field, validate_catalog, compatible_cost,
                     confirmed_funding, relevance_scores, _evaluate_agent5_pairs,
                     summarize_agent5_pair_predictions, apply_agent5_source_enrichments):
        digest.update(_ev_code_bytes(function.__code__))
    digest.update(scholarship_catalog.to_json(orient="records").encode())
    for path in (EVALUATION_ROOT / "data/raw/student_profiles/student_profiles_anonymized.csv",
                 A5_DATA / "agent5_evaluation_states.json", A5_DATA / "agent5_source_enrichments.csv"):
        digest.update(_ev_sha(path).encode() if path.exists() else b"absent")
    return digest.hexdigest()

def freeze_agent5_eligibility_version(version):
    if not re.fullmatch(r"[A-Za-z0-9_-]+", version):
        raise ValueError("Use a unique simple version name.")
    _ev_write_new(A5_RUNS / f"{version}.freeze.json",
                 {"version": version, "fingerprint": _a5_fingerprint(), "frozen_at": _ev_now()})
    print("Eligibility version frozen; use run_agent5_eligibility_test_once explicitly.")

def run_agent5_eligibility_test_once(version, *, human_sources_verified=False):
    if not human_sources_verified:
        raise ValueError("Confirm independent human judgments checked against official pages.")
    freeze = _ev_json.loads((A5_RUNS / f"{version}.freeze.json").read_text(encoding="utf-8"))
    if freeze["fingerprint"] != _a5_fingerprint():
        raise ValueError("Agent, profiles, catalog or contexts changed after freeze.")
    judgments, profiles, contexts, catalog = _agent5_evaluation_inputs()
    groups = sorted(set(judgments.loc[judgments.split.eq("test"), "student_id"]))
    if not groups:
        raise ValueError("No test students.")
    for path in A5_RUNS.glob("*.consumed.json"):
        if set(groups) & set(_ev_json.loads(path.read_text(encoding="utf-8"))["student_ids"]):
            raise ValueError("Test students already consumed; collect a fresh test cohort.")
    _ev_write_new(A5_RUNS / f"{version}.consumed.json",
                 {"student_ids": groups, "consumed_at": _ev_now(),
                  "judgments_sha256": _ev_sha(A5_DATA / "agent5_eligibility_judgments.csv")})
    result = _evaluate_agent5_pairs(judgments, profiles, contexts, catalog, "test", 200)
    report = {**result["report"], "version": version, "fingerprint": freeze["fingerprint"],
              "completed_at": _ev_now()}
    _ev_write_new(A5_RUNS / f"{version}.report.json", report)
    result["predictions"].to_json(A5_RUNS / f"{version}.predictions.json", orient="records", indent=2)
    print(report)
    return result

# Do not turn all-unknown behavior into 100% accuracy.
_abstention_fixture = pd.DataFrame([
    {"student_id": "fixture-a", "agent_decision": "unknown", "human_decision": "eligible", "fully_recommended": False},
    {"student_id": "fixture-b", "agent_decision": "unknown", "human_decision": "cannot_determine", "fully_recommended": False}])
_abstention_report = summarize_agent5_pair_predictions(_abstention_fixture)
assert _abstention_report["selective_accuracy"] is None
assert _abstention_report["decision_coverage"] == 0 and _abstention_report["unknown_share"] == 1
print("Eligibility evaluator ready. All-unknown denominator check passed; no accuracy claim from fixtures.")
def _ev_code_bytes(code):
    """Ignore transient kernel cell filenames, retain executable constants and bytecode."""
    import marshal, types
    def canonical(item):
        constants = tuple(canonical(value) if isinstance(value, types.CodeType) else value
                          for value in item.co_consts)
        return item.replace(co_filename="<evaluation>", co_firstlineno=1, co_consts=constants)
    return marshal.dumps(canonical(code))


Eligibility evaluator ready. All-unknown denominator check passed; no accuracy claim from fixtures.


## Executed scholarship decision checks

The cases below are explicitly fictional rule-contract examples. Expected decisions and
gaps are specified before calling the public agent. The resulting percentage is a
**functional check pass rate**, not scholarship eligibility accuracy on real students.
Eligible applicants with unresolved monetary details remain eligible for review while
receiving no confirmed recommendation. Every case and prediction is displayed.

In [10]:
def run_agent5_prediction_checks():
    base_catalog = fixture_catalog.copy(deep=True)
    base_state = deepcopy(fixture_state)
    base_state['finance'][0]['cost_period'] = 'annual'
    cases = []
    def add(name, expected, recommended, gap=20000, profile=None, catalog=None,
            remove_profile=(), date='2026-10-20', awards=None):
        state, records = deepcopy(base_state), base_catalog.copy(deep=True)
        state['profile'].update(profile or {})
        for key in remove_profile:
            state['profile'].pop(key, None)
        for key, value in (catalog or {}).items():
            records[key] = value
        if awards is not None:
            state['confirmed_awards'] = awards
        cases.append((name, state, records, date, expected, recommended, gap))
    add('Meets all stated rules', 'eligible', True)
    add('GPA below explicit cutoff', 'ineligible', False, profile={'gpa': 7.0})
    add('Missing GPA scale', 'unknown', False, remove_profile=('gpa_scale',))
    add('Expired deadline', 'ineligible', False, date='2027-01-02')
    add('Missing nationality', 'unknown', False, remove_profile=('nationality',))
    add('Indian demonym matches India', 'eligible', True, profile={'nationality': 'Indian'})
    add('Explicit nationality mismatch', 'ineligible', False, profile={'nationality': 'Germany'})
    add('Related field requires review', 'unknown', False, profile={'field_of_study': 'Computer Science and Engineering'})
    add('MS-only award excludes MBA', 'ineligible', False, profile={'target_degree': 'MBA'}, catalog={'eligible_degrees': 'MS'})
    add('Missing required merit score', 'unknown', False, catalog={'min_profile_strength_score': 75})
    add('Merit 82 clears cutoff 75', 'eligible', True, profile={'profile_strength_score': 82}, catalog={'min_profile_strength_score': 75})
    add('Merit 72 fails cutoff 75', 'ineligible', False, profile={'profile_strength_score': 72}, catalog={'min_profile_strength_score': 75})
    add('Unstated award period withheld', 'eligible', False, catalog={'amount_period': ''})
    add('Award currency mismatch withheld', 'eligible', False, catalog={'currency': 'EUR'})
    add('Unresolved extra rules', 'unknown', False, catalog={'_raw_rules_need_review': True})
    add('Missing budget means unknown gap', 'eligible', True, gap=None, remove_profile=('budget_annual',))
    confirmed = {**fixture_award, 'exclusive_group': '', 'annual_amount': 6000, 'status': 'confirmed'}
    add('Confirmed annual award reduces gap', 'eligible', True, gap=14000, awards=[confirmed])
    add('Applied award does not reduce gap', 'eligible', True, awards=[{**confirmed, 'status': 'applied'}])
    records = []
    for name, state, catalog, date, expected, recommended, expected_gap in cases:
        result = scholarship_agent(state, catalog, as_of=date)['scholarships'][0]
        decisions = result['recommendations'] + result['eligibility_review']
        assert len(decisions) == 1
        predicted = decisions[0]['eligibility_match']
        gap = result['funding_gap']
        passed = (predicted == expected and bool(result['recommendations']) == recommended
                  and (gap == expected_gap if expected_gap is None else gap is not None and abs(gap-expected_gap) < 1e-8))
        records.append({'case': name, 'expected': expected, 'predicted': predicted,
                        'recommended': bool(result['recommendations']), 'funding_gap': gap, 'passed': passed})
    return pd.DataFrame(records)

agent5_prediction_checks = run_agent5_prediction_checks()
print(agent5_prediction_checks.to_string(index=False))
agent5_functional_result = {'passed': int(agent5_prediction_checks.passed.sum()),
    'cases': len(agent5_prediction_checks), 'pass_rate': float(agent5_prediction_checks.passed.mean()),
    'evaluation_type': 'fictional_functional_checks'}
assert agent5_prediction_checks.passed.all(), 'A displayed scholarship contract case failed.'

                              case   expected  predicted  recommended  funding_gap  passed
            Meets all stated rules   eligible   eligible         True      20000.0    True
         GPA below explicit cutoff ineligible ineligible        False      20000.0    True
                 Missing GPA scale    unknown    unknown        False      20000.0    True
                  Expired deadline ineligible ineligible        False      20000.0    True
               Missing nationality    unknown    unknown        False      20000.0    True
      Indian demonym matches India   eligible   eligible         True      20000.0    True
     Explicit nationality mismatch ineligible ineligible        False      20000.0    True
     Related field requires review    unknown    unknown        False      20000.0    True
        MS-only award excludes MBA ineligible ineligible        False      20000.0    True
      Missing required merit score    unknown    unknown        False      20000.0    True

## Final accuracy and catalog counts

Frozen reports are displayed without rerunning test predictions.

In [11]:
print("AGENT 5 - FINAL EXECUTED RESULTS")
agent5_accuracy_summary = {"eligibility_selective_accuracy": None, "decision_coverage": None,
                          "funding_support_test_accuracy": None, "selective_96_supported": False}
if scholarship_catalog is not None:
    print(f"Uploaded scholarships: {len(scholarship_catalog):,}")
    print("Illustrative profile, full catalog:")
    print(pd.DataFrame([agent5_catalog_demo["scholarships"][0]["catalog_counts"]]).to_string(index=False))
    print("These are completeness counts for an illustrative profile, not accuracy on real students.")
_reports = sorted(A5_RUNS.glob("*.report.json"), key=lambda p: p.stat().st_mtime)
if _reports:
    _report = _ev_json.loads(_reports[-1].read_text(encoding="utf-8"))
    if _report["fingerprint"] == _a5_fingerprint():
        print("Saved frozen eligibility test report (not re-run):", _report)
        agent5_accuracy_summary.update(eligibility_selective_accuracy=_report["selective_accuracy"],
            decision_coverage=_report["decision_coverage"], selective_96_supported=_report["selective_96_supported"])
    else:
        print("Saved eligibility report belongs to a different version; current version remains unverified.")
else:
    print("Held-out eligibility selective accuracy / decision coverage / unknown share: NOT MEASURED.")
print("Selective 96% confidence target:", "SUPPORTED" if agent5_accuracy_summary["selective_96_supported"] else "NOT VERIFIED")
print("Funding-gap arithmetic: functional checks PASSED; this is not predictive accuracy.")
print("Funding-support classifier: NOT MEASURED here; separate independent outcomes and explicit evaluation required.")
print("Real source review is required to fill missing units and resolve eligibility text.")
print(agent5_contract_status)
print("EXECUTED FUNCTIONAL PREDICTIONS (not eligibility accuracy):")
print(f"Pass rate: {agent5_functional_result['pass_rate']:.2%} ({agent5_functional_result['passed']}/{agent5_functional_result['cases']} cases)")


AGENT 5 - FINAL EXECUTED RESULTS
Uploaded scholarships: 9,673
Illustrative profile, full catalog:
 examined  fully_recommendable  needs_review  ineligible
     9673                    0           630        9043
These are completeness counts for an illustrative profile, not accuracy on real students.
Held-out eligibility selective accuracy / decision coverage / unknown share: NOT MEASURED.
Selective 96% confidence target: NOT VERIFIED
Funding-gap arithmetic: functional checks PASSED; this is not predictive accuracy.
Funding-support classifier: NOT MEASURED here; separate independent outcomes and explicit evaluation required.
Real source review is required to fill missing units and resolve eligibility text.
PASSED: legacy annual budget, 0-100 merit, country propagation and explicit nonannual budgets.
EXECUTED FUNCTIONAL PREDICTIONS (not eligibility accuracy):
Pass rate: 100.00% (18/18 cases)
